# 📝 Atomic Step 7b: Report Generation & PDF Serialization (Local Gemma-3n-e4b)

This notebook runs entirely offline/locally without any external Gemini API dependencies. It uses the multimodal language model **Gemma 3n E4B** (`google/gemma-3n-e4b-it`) to:
1. **Convert/Transliterate** Gurmukhi script Punjabi transcripts into Devanagari script (Hindi/Punjabi-in-Devanagari).
2. **Extract Structured Interaction Metadata**: Scans the transcript to find Date, Day, Sarpanch, Panchayat, Phone number, Event location, Block, District, Farmer counts, Coordinator, Narrations, Summaries, and a comprehensive numbered list of **Key Challenges (10-11 items)**.
3. **Serialize to PDF**: Compiles the report into a premium-looking PDF document using the `markdown-pdf` library, saving it directly to Google Drive.

In [ ]:
# 1. Uninstall incompatible torchao to prevent conflicts
!pip uninstall -y -q torchao

# 2. Install Transformers, Accelerate, openpyxl, python-docx, reportlab, and loguru
!pip install -q transformers accelerate openpyxl python-docx reportlab loguru --prefer-binary
print("[SUCCESS] Dependencies installed!")

In [ ]:
import torch

# Set device and dtype
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.bfloat16 if "cuda" in device else torch.float32
print(f"[INFO] Using device: {device} | Dtype: {torch_dtype}")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except Exception:
    print("[INFO] Already mounted or skipped.")

# @markdown ### 📁 Transcript & Folder Inputs
input_transcript_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files/FullNarration_Gagg.m4a" # @param {type:"string"}
report_output_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Reports" # @param {type:"string"}

# @markdown ### 🤖 Model Configs
gemma_model_id = "google/gemma-3n-e2b-it" # @param {type:"string"}

import os
import glob
from pathlib import Path

# Determine if input_transcript_folder is a file or a folder
input_path = Path(input_transcript_folder)
json_path = None
audio_name_only = ""

if input_path.is_file() or input_path.suffix.lower() in ['.m4a', '.mp3', '.wav', '.mp4']:
    # It's an audio/video file! Locate matching JSON transcript in the same folder or parent/adjacent folders
    parent_dir = input_path.parent
    audio_name_only = input_path.stem
    # 1. Look in the same folder
    target_json = parent_dir / f"{audio_name_only}_diarized_transcript.json"
    if target_json.exists():
        json_path = str(target_json)
    else:
        # 2. Look in subdirectories of parent
        matches = list(parent_dir.glob(f"**/{audio_name_only}_diarized_transcript.json"))
        if not matches:
            # 3. Look in parent's parent
            matches = list(parent_dir.parent.glob(f"**/{audio_name_only}_diarized_transcript.json"))
        if matches:
            json_path = str(matches[0])
else:
    # It's a folder, search inside it as before
    json_files = glob.glob(os.path.join(input_transcript_folder, "*_diarized_transcript.json"))
    if json_files:
        json_path = json_files[0]
        audio_name_only = os.path.basename(json_path).replace("_diarized_transcript.json", "")

# Default report_output_folder to input folder if not specified
if not report_output_folder.strip():
    report_output_folder = str(input_path.parent) if input_path.is_file() else input_transcript_folder

if not json_path or not os.path.exists(json_path):
    print(f"[ERROR] Transcript JSON file not found for: '{input_transcript_folder}'")
else:
    os.makedirs(report_output_folder, exist_ok=True)
    print(f"[SUCCESS] Validated paths. Found transcript: {os.path.basename(json_path)}")
    print(f"Reports will be saved to: {report_output_folder}")

In [ ]:
# @markdown ### 📁 Transcript & Folder Inputs
input_transcript_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Diarization_Transcripts/MarauliKhurad1" # @param {type:"string"}
report_output_folder = "" # @param {type:"string"}

# @markdown ### 🤖 Model Configs
gemma_model_id = "google/gemma-3n-e4b-it" # @param {type:"string"}

import os
import glob

# If output folder not specified, default to input folder
if not report_output_folder.strip():
    report_output_folder = input_transcript_folder

# Locate transcript JSON
json_files = glob.glob(os.path.join(input_transcript_folder, "*_diarized_transcript.json"))

if not os.path.exists(input_transcript_folder):
    print(f"[ERROR] Transcript folder not found at: '{input_transcript_folder}'")
elif not json_files:
    print(f"[ERROR] No transcript JSON file found inside folder: '{input_transcript_folder}'")
else:
    json_path = json_files[0]
    audio_name_only = os.path.basename(json_path).replace("_diarized_transcript.json", "")
    os.makedirs(report_output_folder, exist_ok=True)
    print(f"[SUCCESS] Validated paths. Found transcript: {os.path.basename(json_path)}")
    print(f"Reports will be saved to: {report_output_folder}")

import json
import re
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

if 'json_path' in locals() and os.path.exists(json_path):
    # Load HF Token
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except (ImportError, ValueError):
        HF_TOKEN = os.environ.get("HF_TOKEN")
        
    if not HF_TOKEN:
        print("[WARNING] HF_TOKEN not found in environment. Model download might fail if restricted.")

    # Load transcript entries
    print(f"Loading transcript file: '{json_path}'")
    with open(json_path, "r", encoding="utf-8") as f:
        diarized_transcript_entries = json.load(f)
        
    # Format transcript text
    transcript_text = ""
    for entry in diarized_transcript_entries:
        transcript_text += f"{entry['time']} {entry['speaker']}: {entry['text']}\n"
        
    # Load Model and Processor
    print(f"Loading processor and model for {gemma_model_id}...")
    processor = AutoProcessor.from_pretrained(gemma_model_id, token=HF_TOKEN)
    model = AutoModelForMultimodalLM.from_pretrained(
        gemma_model_id,
        token=HF_TOKEN,
        torch_dtype=torch_dtype,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    model.eval()
    print(f"[SUCCESS] Gemma model {gemma_model_id} loaded successfully!")
    
    # Construct detailed structured JSON prompt
    prompt_text = f"""
You are provided with a sequence-based speaker-diarized Punjabi transcript (written in Gurmukhi script).
Please analyze the transcript and return your findings STRICTLY as a JSON object matching the schema below.
Ensure you do not invent information; if a metadata field is not mentioned, use null, \"\", or an empty array as appropriate.

JSON Schema:
{{
  \"metadata\": {{
    \"date\": \"YYYY-MM-DD\", // Date of the interaction. Use format YYYY-MM-DD.
    \"village\": \"...\", // Village name.
    \"sarpanch_name\": \"...\", // Name of the Sarpanch.
    \"panchayat\": \"...\", // Panchayat name.
    \"sarpanch_phone\": \"...\", // Phone number of the Sarpanch.
    \"event_location\": \"...\", // Location of the event.
    \"block\": \"...\", // Block name.
    \"district\": \"...\", // District name.
    \"coordinator_name\": \"...\", // Name of the coordinator.
    \"reporting_manager_name\": \"...\", // Name of the reporting manager.
    \"event_start_time\": \"HH:MM\", // Event start time.
    \"event_end_time\": \"HH:MM\", // Event end time.
    \"farmer_counts\": {{
      \"male\": 0, // Number of male farmers.
      \"female\": 0 // Number of female farmers.
    }}
  }},
  \"participants\": {{
    \"total_count\": 0, // Total number of farmer participants.
    \"farmer_names\": [] // List of specific individual names of farmer participants. Exclude generic titles like \"Farmer\", \"Speaker\", \"Coordinator\".
  }},
  \"narration\": {{
    \"detailed_narration\": \"...\", // A comprehensive chronological description of the discussion, who said what, and the flow of conversation (2-3 detailed paragraphs).
    \"summary\": \"...\" // A concise summary of the discussion (1 paragraph).
  }},
  \"key_challenges\": [], // Numbered/list of key agricultural challenges mentioned by farmers (provide a comprehensive list, at least 10 to 11 challenges if present).
  \"farmer_questions\": [], // Rephrased agricultural questions or concerns asked by farmers, written as clear full sentences in English. Exclude non-agricultural questions.
  \"terminology_mapping\": [
    {{
      \"Crop\": \"...\", // Crop name (e.g., Wheat, Paddy)
      \"Local Name\": \"...\", // Punjabi Unicode Local/Dialect name (e.g. ਪੀਲੀ ਕੁੰਗੀ (Peeli Kungi))
      \"Standard Name\": \"...\", // Standard common name in English (e.g. Yellow Rust)
      \"Scientific Name\": \"...\", // Scientific name (e.g. Puccinia striiformis)
      \"Language\": \"...\" // Language/Dialect (e.g. Punjabi)
    }}
  ],
  \"conclusion\": \"...\" // A comprehensive 2-3 paragraph conclusion summarizing the overall meeting purpose, key agricultural issues, and actionable next steps.
}}

Return ONLY valid JSON. No conversational text. Do not wrap the JSON in markdown code blocks unless requested.

--- Transcript ---
{transcript_text}
"""

    # Prepare message for Gemma 3n
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_text},
            ],
        },
    ]
    
    print("Preparing prompt and generating insights locally via Gemma 3n...")
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]
    
    try:
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
                do_sample=False,
                repetition_penalty=1.1
            )
            
        response_ids = generated_ids[0][input_len:]
        insights_json_str = processor.decode(response_ids, skip_special_tokens=True).strip()
        
        # Save JSON file
        json_insights_path = os.path.join(report_output_folder, f"{audio_name_only}_gemma_analysis.json")
        with open(json_insights_path, "w", encoding="utf-8") as f:
            f.write(insights_json_str)
            
        print(f"\n[SUCCESS] Devanagari translation & insights successfully stored as JSON at: '{json_insights_path}'")
    except Exception as e:
        print(f"[ERROR] Local Gemma generation failed: {e}")

In [ ]:
## 📄 Step 2: Format and Generate Reports (PDF, Excel, Word)

import os
import sys
import json
import re
import shutil
import logging
from pathlib import Path
from datetime import datetime
from types import ModuleType

if 'insights_json_str' in locals() and insights_json_str:
    # 1. Clean and parse Gemma's JSON output
    def parse_json_response(response_text: str) -> dict:
        # Find JSON block markers
        match = re.search(r'```json\\s*(.*?)\\s*```', response_text, re.DOTALL | re.IGNORECASE)
        if match:
            json_str = match.group(1).strip()
        else:
            start = response_text.find('{')
            end = response_text.rfind('}') + 1
            if start != -1 and end != -1:
                json_str = response_text[start:end]
            else:
                json_str = response_text.strip()
        return json.loads(json_str)

    try:
        report_data = parse_json_response(insights_json_str)
        
        # Convert date string to datetime object if possible for PDFReportGenerator
        meta = report_data.get('metadata', {})
        date_str = meta.get('date')
        if date_str:
            try:
                meta['date'] = datetime.strptime(date_str, "\%Y-\%m-\%d")
            except Exception:
                try:
                    meta['date'] = datetime.strptime(date_str, "\%d-\%m-\%Y")
                except Exception:
                    pass
                    
        # 2. Setup paths and mocks to import App/cli-tool reports module
        repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
        cli_tool_path = os.path.abspath(os.path.join(repo_root, "App/cli-tool"))
        if cli_tool_path not in sys.path:
            sys.path.insert(0, cli_tool_path)
            
        # Mock config settings
        config_mock = ModuleType('config')
        class SettingsMock:
            log_level = "INFO"
            log_file = os.path.join(repo_root, "logs/pipeline.log")
        config_mock.settings = SettingsMock()
        sys.modules['config'] = config_mock
        
        # Mock src.core.utils logger
        utils_mock = ModuleType('src.core.utils')
        utils_mock.log = logging.getLogger('reports_notebook')
        sys.modules['src.core.utils'] = utils_mock
        
        # Mock src.modules.analysis transliterate
        analysis_mock = ModuleType('src.modules.analysis')
        def transliterate_punjabi_to_english(text: str) -> str:
            if not text: return text
            transliteration_map = {
                'ਅ': 'a', 'ਆ': 'aa', 'ਇ': 'i', 'ਈ': 'ee', 'ਉ': 'u', 'ਊ': 'oo', 'ਏ': 'e', 'ਐ': 'ai', 'ਓ': 'o', 'ਔ': 'au',
                'ਕ': 'k', 'ਖ': 'kh', 'ਗ': 'g', 'ਘ': 'gh', 'ਙ': 'ng',
                'ਚ': 'ch', 'ਛ': 'chh', 'ਜ': 'j', 'ਝ': 'jh', 'ਞ': 'ny',
                'ਟ': 't', 'ਠ': 'th', 'ਡ': 'd', 'ਢ': 'dh', 'ਣ': 'n',
                'ਤ': 't', 'ਥ': 'th', 'ਦ': 'd', 'ਧ': 'dh', 'ਨ': 'n',
                'ਪ': 'p', 'ਫ': 'ph', 'ਬ': 'b', 'ਭ': 'bh', 'ਮ': 'm',
                'ਯ': 'y', 'ਰ': 'r', 'ਲ': 'l', 'ਵ': 'v', 'ੜ': 'r',
                'ਸ': 's', 'ਹ': 'h', 'ਸ਼': 'sh', 'ਖ਼': 'kh', 'ਗ਼': 'gh', 'ਜ਼': 'z', 'ਫ਼': 'f',
                'ਾ': 'aa', 'ਿ': 'i', 'ਈ': 'ee', 'ੁ': 'u', 'ੂ': 'oo', 'ੇ': 'e', 'ੈ': 'ai', 'ੋ': 'o', 'ੌ': 'au',
                'ੰ': 'n', 'ਂ': 'n', 'ੱ': '', '੍': '',
                'क': 'k', 'ख': 'kh', 'ग': 'g', 'घ': 'gh', 'ङ': 'ng',
                'च': 'ch', 'छ': 'chh', 'ज': 'j', 'झ': 'jh', 'ञ': 'ny',
                'ट': 't', 'ठ': 'th', 'ड': 'd', 'ढ': 'dh', 'ण': 'n',
                'त': 't', 'थ': 'th', 'द': 'd', 'ध': 'dh', 'न': 'n',
                'प': 'p', 'फ': 'ph', 'ब': 'b', 'भ': 'bh', 'म': 'm',
                'य': 'y', 'र': 'r', 'ल': 'l', 'व': 'v', 'श': 'sh', 'ष': 'sh', 'स': 's', 'ह': 'h',
                'अ': 'a', 'आ': 'aa', 'इ': 'i', 'ई': 'ee', 'ਉ': 'u', 'ਊ': 'oo', 'ਏ': 'e', 'ਐ': 'ai', 'ਓ': 'o', 'ਔ': 'au',
                'ਾ': 'aa', 'ਿ': 'i', 'ੀ': 'ee', 'ੁ': 'u', 'ੂ': 'oo', 'ੇ': 'e', 'ੈ': 'ai', 'ੋ': 'o', 'ੌ': 'au', '्': '',
                'ं': 'n', 'ँ': 'n',
            }
            result = []
            for char in text:
                result.append(transliteration_map.get(char, char))
            transliterated = ''.join(result).strip()
            words = transliterated.split()
            capitalized_words = []
            for word in words:
                word_lower = word.lower()
                if word_lower == 'singh': capitalized_words.append('Singh')
                elif word_lower == 'kaur': capitalized_words.append('Kaur')
                elif word_lower == 'kumar': capitalized_words.append('Kumar')
                else: capitalized_words.append(word_lower.capitalize())
            return ' '.join(capitalized_words)
        analysis_mock.transliterate_punjabi_to_english = transliterate_punjabi_to_english
        sys.modules['src.modules.analysis'] = analysis_mock

        # Copy fonts locally to resolve assets/fonts path in PDFReportGenerator
        src_font = Path(cli_tool_path) / "assets/fonts/FreeSans.ttf"
        dest_font = Path("assets/fonts/FreeSans.ttf")
        if src_font.exists() and not dest_font.exists():
            dest_font.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(src_font, dest_font)
            print("[INFO] Copied FreeSans.ttf locally for ReportLab registration.")

        # 3. Import and execute report generation modules
        from src.modules.reports import excel_generator, pdf_generator, word_generator
        
        print("Generating reports using App/cli-tool formats...")
        
        # Excel Report
        excel_path = Path(report_output_folder) / f"{audio_name_only}_report.xlsx"
        excel_generator.create_report(report_data, excel_path)
        print(f"[SUCCESS] Excel report generated: {excel_path}")
        
        # PDF Report
        pdf_path = Path(report_output_folder) / f"{audio_name_only}_report.pdf"
        pdf_generator.create_report(report_data, pdf_path)
        print(f"[SUCCESS] PDF report generated: {pdf_path}")
        
        # Word Report
        word_path = Path(report_output_folder) / f"{audio_name_only}_report.docx"
        word_generator.create_report(report_data, word_path)
        print(f"[SUCCESS] Word report generated: {word_path}")

    except Exception as e:
        print(f"[ERROR] Report generation process failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("[ERROR] No structured JSON content available to generate reports. Make sure Step 1 ran successfully.")

In [ ]:
from markdown_pdf import MarkdownPdf, Section

if 'insights_md' in locals() and insights_md:
    pdf_insights_path = os.path.join(report_output_folder, f"{audio_name_only}_gemma_report.pdf")
    print(f"Generating premium PDF report at: '{pdf_insights_path}'")
    
    try:
        # Create PDF object
        pdf = MarkdownPdf(toc_level=2)
        
        # Add content section
        pdf.add_section(Section(insights_md))
        
        # Save PDF
        pdf.save(pdf_insights_path)
        print(f"\n[SUCCESS] Report successfully serialized to PDF: '{pdf_insights_path}'")
    except Exception as e:
        print(f"[ERROR] PDF generation failed: {e}")
else:
    print("[ERROR] No report content available to generate PDF. Make sure Step 1 ran successfully.")